<div style="background:#E9FFF6; color:#440404; padding:8px; border-radius: 4px; text-align: center; font-weight: 500;">IFN619 - Data Analytics for Strategic Decision Makers</div>

# IFN619 :: C1-UnstructuredAnalytics

For this session, the focus will be on analysis of unstructured text. However, the thinking required is similar to approaches to analysing images, video, sound and other unstructured data. Primarily, the analysis is based on the notion that there are useful patterns in the unstructured data which can be obtained mathematically. By converting the data to a mathematical structure, various algorithms can be applied to the structure with the aim of identifying patterns. 

In the case of the `topic modelling` approaches below, the techniques are *stochastic* - that is they include a level of randomness and mathematically identify the probability or *likelihood* that a feature might be important. Thus, they are never deterministic, repeatable or 100% accurate, and their use needs to be mediated by a more pragmatic *useful or not* approach, rather than *right or wrong*.

In [1]:
# Import the necessary libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
import pandas as pd
import json
import random

### Accessing the data via The Guardian API

See the `Accessing_the_Guardian_API.ipynb` notebook file for details on getting the data. **Note:** This approach may be used for additional data for Assignment 2.

### Read in pre-saved data

To save time, we're loading in pre-saved data that was fetched using the Guardian API.

In [2]:
# Load the data - articles from The Guardian about the Winter Olympics
file_path = "data/"
file_name = "winter_olympics_articles.json"

with open(f"{file_path}{file_name}",'r', encoding='utf-8') as fp:
    articles = json.load(fp)

print(f"Loaded {len(articles)} articles from {file_name}")

Loaded 13 articles from winter_olympics_articles.json


Each dictionary entry includes the *title [date]* as `key` and the *body text* from the article as `value`.

So the values gives us a list of documents that we can analyse.

In [3]:
# Get a list of documents
documents = list(articles.values())

# View first 400 characters of the 1st document
documents[0][:400]

'When do the 2026 Winter Olympics start? The 2026 Winter Olympics officially open in the early hours of Saturday 7 February, Australian time, with the opening ceremony at Milan’s San Siro stadium. The Games run for two weeks, culminating in the closing ceremony on 23 February in Verona at the same time of 6am AEDT. Several sports with packed schedules, including curling and ice hockey, begin a coup'

### Term Count and TF/IDF as vector inputs for Topic Modelling

Recall from B3 that we created bag of words vectors using `CountVectorizer` and `TfidfVectorizer`.

In this lecture will use these vectors as inputs for our topic modelling algorithms.

All of these analyses, approach the document as a [Bag of Words](https://en.wikipedia.org/wiki/Bag-of-words_model) model. In this approach, the order of the words don't matter. A popular approach that takes into account order is [Word embedding](https://en.wikipedia.org/wiki/Word_embedding). This session does not explore word embedding.

In [4]:
# Only count terms that in maximum of 80% of documents, and a minimum of 2 documents. 
# Count a maximum of 10000 terms, and remove common english stop words
count_vectorizer = CountVectorizer(max_df=0.80,min_df=2,max_features=10000,stop_words="english")
count_dt_matrix = count_vectorizer.fit_transform(articles.values())

In [5]:
# Get the 1000 terms identified during the vectorization process
feature_names = count_vectorizer.get_feature_names_out()
feature_names

array(['000', '10', '10th', '12', '15', '16', '17', '1994', '1998', '20',
       '2002', '2006', '2010', '2022', '2024', '2025', '2026', '20th',
       '21', '23', '25', '26', '27', '28', '30', '31', '50', '53', 'able',
       'accepted', 'according', 'accused', 'achieve', 'acknowledged',
       'acl', 'action', 'actually', 'added', 'adelaide', 'advertiser',
       'advice', 'aedt', 'aerial', 'aerials', 'afternoon', 'age', 'ago',
       'ahead', 'ahmad', 'air', 'alisa', 'allegedly', 'alongside',
       'alpine', 'alps', 'america', 'american', 'amid', 'annual',
       'anthony', 'appeal', 'arms', 'arrived', 'asian', 'assessment',
       'athlete', 'athletes', 'attack', 'attempt', 'attention',
       'australians', 'awarded', 'away', 'backed', 'baff', 'base',
       'based', 'battling', 'bearer', 'beat', 'beating', 'began', 'begin',
       'beijing', 'ben', 'best', 'better', 'bid', 'big', 'biggest', 'bit',
       'blood', 'blow', 'body', 'bradbury', 'brain', 'breadth',
       'breakthrou

In [6]:
# Take a look at the vocabulary which shows the total counts for whole collection
count_vectorizer.vocabulary_

{'2026': np.int64(16),
 'start': np.int64(624),
 'open': np.int64(464),
 'early': np.int64(211),
 'hours': np.int64(321),
 'saturday': np.int64(562),
 'february': np.int64(242),
 'time': np.int64(665),
 'opening': np.int64(465),
 'ceremony': np.int64(119),
 'milan': np.int64(421),
 'run': np.int64(556),
 'weeks': np.int64(712),
 'closing': np.int64(138),
 '23': np.int64(19),
 'aedt': np.int64(41),
 'sports': np.int64(617),
 'including': np.int64(329),
 'ice': np.int64(323),
 'begin': np.int64(82),
 'couple': np.int64(167),
 'days': np.int64(182),
 'mixed': np.int64(429),
 'team': np.int64(652),
 'just': np.int64(355),
 'missed': np.int64(425),
 'qualifying': np.int64(517),
 'despite': np.int64(194),
 'ranked': np.int64(525),
 'december': np.int64(185),
 'won': np.int64(721),
 'italy': np.int64(340),
 'final': np.int64(248),
 'event': np.int64(227),
 'men': np.int64(419),
 'game': np.int64(275),
 '12': np.int64(3),
 'related': np.int64(539),
 'youngest': np.int64(727),
 'olympian': np.i

In [7]:
# Create a new dataframe with the matrix - use titles for the index and terms for the columns
count_df = pd.DataFrame(count_dt_matrix.toarray(), columns=feature_names)
#count_df

Keep a list of the titles of the articles to make it easy to see the headline that relates to the topics. We can always go back to the original documents if we need to.

In [8]:
titles=list(articles.keys())
titles

['Winter Olympics 2026: what you need to know if following from Australia [2026-02-02T14:00:10Z]',
 'Australia’s upward trajectory slips off course as Winter Olympics medal search goes on | Jack Snape [2026-02-12T03:36:06Z]',
 '‘Even more special’: Jakara Anthony dusts off Winter Olympics heartbreak for historic triumph [2026-02-15T01:32:16Z]',
 'A part-time job and DJ gigs helped Lara Hamilton reach the Winter Olympics. Now she wants to put Australia on the map [2026-02-17T14:00:15Z]',
 'Valentino Guseli’s unexpected big air dream ends without a medal in all-or-nothing final: ‘I left it all out there’ [2026-02-08T00:27:13Z]',
 'From Bradbury to Bright: five of Australia’s best Winter Olympic moments | Martin Pegan [2026-02-03T14:00:08Z]',
 'Australia’s youngest Winter Olympian Indra Brown: ‘I just love the feeling of flying’ | Martin Pegan [2026-02-01T14:00:34Z]',
 'How did Australia – better known for its beaches than snow – become a consistent Winter Olympics performer? | Kieran Pen

By selecting a row from the dataframe and sorting the values (counts), we can identify the top 10 terms

In [9]:
# Sample 5 random article numbers
samples = random.sample(range(0,len(count_df)),5)

# View the associated terms
for sample in samples:
    doc = count_df.iloc[sample]
    title = titles[sample]
    top_terms = dict(count_df.iloc[sample].sort_values(ascending=False).head(10))
    print(f"[{sample}] {title}")
    print("\t- Top terms:",top_terms)
    print()

[11] Morning Mail: Lake Cargelligo suspect’s past revealed, supermarket ‘per unit’ prices under scrutiny, shark attack spike [2026-02-18T20:23:22Z]
	- Top terms: {'morning': np.int64(4), 'history': np.int64(3), 'including': np.int64(3), 'new': np.int64(3), 'government': np.int64(3), 'people': np.int64(3), 'day': np.int64(3), 'children': np.int64(3), 'crossword': np.int64(2), 'asian': np.int64(2)}

[1] Australia’s upward trajectory slips off course as Winter Olympics medal search goes on | Jack Snape [2026-02-12T03:36:06Z]
	- Top terms: {'team': np.int64(7), 'anthony': np.int64(7), 'final': np.int64(5), 'competition': np.int64(5), 'medals': np.int64(4), 'training': np.int64(4), 'event': np.int64(4), '000': np.int64(3), 'air': np.int64(3), 'big': np.int64(3)}

[7] How did Australia – better known for its beaches than snow – become a consistent Winter Olympics performer? | Kieran Pender [2026-02-17T01:14:10Z]
	- Top terms: {'medals': np.int64(8), 'moguls': np.int64(7), 'sports': np.int64(

#### Create a top10 terms dataframe

Using the index from the documents, create a dataframe that can hold the top10 terms for each document. We also include columns for our other analysis (tfidf, lda, nmf)

In [10]:
# Create a dataframe to hold top terms for each analysis type
terms_df = pd.DataFrame(index=count_df.index,columns=['title','count','tfidf','lda','nmf'])
terms_df['title'] = titles
terms_df

,title,count,tfidf,lda,nmf
0,Winter Olympics 2026: what you need to know if...,NaN,NaN,NaN,NaN
1,Australia’s upward trajectory slips off course...,NaN,NaN,NaN,NaN
2,‘Even more special’: Jakara Anthony dusts off ...,NaN,NaN,NaN,NaN
3,A part-time job and DJ gigs helped Lara Hamilt...,NaN,NaN,NaN,NaN
4,Valentino Guseli’s unexpected big air dream en...,NaN,NaN,NaN,NaN
5,From Bradbury to Bright: five of Australia’s b...,NaN,NaN,NaN,NaN
6,Australia’s youngest Winter Olympian Indra Bro...,NaN,NaN,NaN,NaN
7,How did Australia – better known for its beach...,NaN,NaN,NaN,NaN
8,‘Put the blinders on’: how Jakara Anthony can ...,NaN,NaN,NaN,NaN
9,Ice in his veins: Australian skier Cooper Wood...,NaN,NaN,NaN,NaN


Populate the count column with data created by the count vectorizer.

In [11]:
#For each doc, get the 10 columns with the largest counts
for idx in terms_df.index:
    counts = dict(count_df.loc[idx].sort_values(ascending=False).head(10))
    #print(counts)
    terms_df.at[idx,'count'] = list(counts.keys()) # Just the list of terms

terms_df

,title,count,tfidf,lda,nmf
0,Winter Olympics 2026: what you need to know if...,"[2026, ice, new, athletes, aedt, day, ski, wat...",NaN,NaN,NaN
1,Australia’s upward trajectory slips off course...,"[team, anthony, final, competition, medals, tr...",NaN,NaN,NaN
2,‘Even more special’: Jakara Anthony dusts off ...,"[moguls, anthony, dual, single, event, just, f...",NaN,NaN,NaN
3,A part-time job and DJ gigs helped Lara Hamilt...,"[hamilton, ski, skimo, just, running, says, sk...",NaN,NaN,NaN
4,Valentino Guseli’s unexpected big air dream en...,"[final, guseli, said, time, score, air, best, ...",NaN,NaN,NaN
5,From Bradbury to Bright: five of Australia’s b...,"[bradbury, event, high, bronze, skiing, way, s...",NaN,NaN,NaN
6,Australia’s youngest Winter Olympian Indra Bro...,"[just, indra, brown, time, says, want, halfpip...",NaN,NaN,NaN
7,How did Australia – better known for its beach...,"[medals, moguls, sports, team, funding, olympi...",NaN,NaN,NaN
8,‘Put the blinders on’: how Jakara Anthony can ...,"[anthony, moguls, says, ski, win, field, seaso...",NaN,NaN,NaN
9,Ice in his veins: Australian skier Cooper Wood...,"[woods, said, just, cooper, lot, final, win, t...",NaN,NaN,NaN


The [TF/IDF](https://en.wikipedia.org/wiki/Tf–idf) algorithm takes the term frequencies for a document and divides them by the frequencies of the terms in the whole collection.


In [12]:
# Only count terms that in maximum of 80% of documents, and a minimum of 2 documents. 
# Count a maximum of 10000 terms, and remove common english stop words
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.80, min_df=2, max_features=10000, stop_words="english"
)

In [13]:
# Get the document vectors
tfidf_dt_matrix = tfidf_vectorizer.fit_transform(articles.values())

# Display the vector for the first document
tfidf_dt_matrix.toarray()[0]

array([0.05782947, 0.05782947, 0.        , 0.13860385, 0.07100469,
       0.03854199, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.05782947, 0.        , 0.10650704, 0.        ,
       0.        , 0.19692714, 0.        , 0.        , 0.10256164,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.05782947, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.2313179 , 0.        , 0.        , 0.        ,
       0.        , 0.042051  , 0.09240257, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.04620128, 0.05782947,
       0.05782947, 0.        , 0.        , 0.        , 0.03282119,
       0.        , 0.        , 0.        , 0.05782947, 0.        ,
       0.03854199, 0.17751174, 0.        , 0.05782947, 0.        ,
       0.042051  , 0.        , 0.        , 0.        , 0.     

In [14]:
# Create a dataframe for TF/IDF
tfidf_df = pd.DataFrame(tfidf_dt_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
#tfidf_df

In [15]:
# Update the terms dataframe with TF/IDF
for idx in terms_df.index:
    tfidf = dict(tfidf_df.loc[idx].sort_values(ascending=False).head(10))
    #print(counts)
    terms_df.at[idx,'tfidf'] = list(tfidf.keys()) 

terms_df

,title,count,tfidf,lda,nmf
0,Winter Olympics 2026: what you need to know if...,"[2026, ice, new, athletes, aedt, day, ski, wat...","[ice, aedt, watch, 2026, athletes, table, moun...",NaN,NaN
1,Australia’s upward trajectory slips off course...,"[team, anthony, final, competition, medals, tr...","[anthony, team, competition, air, snowboarder,...",NaN,NaN
2,‘Even more special’: Jakara Anthony dusts off ...,"[moguls, anthony, dual, single, event, just, f...","[moguls, anthony, dual, single, course, event,...",NaN,NaN
3,A part-time job and DJ gigs helped Lara Hamilt...,"[hamilton, ski, skimo, just, running, says, sk...","[hamilton, skimo, running, ski, mountaineering...",NaN,NaN
4,Valentino Guseli’s unexpected big air dream en...,"[final, guseli, said, time, score, air, best, ...","[guseli, final, air, 50, landing, score, said,...",NaN,NaN
5,From Bradbury to Bright: five of Australia’s b...,"[bradbury, event, high, bronze, skiing, way, s...","[bradbury, smith, high, bronze, city, event, s...",NaN,NaN
6,Australia’s youngest Winter Olympian Indra Bro...,"[just, indra, brown, time, says, want, halfpip...","[indra, want, just, brown, says, ve, time, fam...",NaN,NaN
7,How did Australia – better known for its beach...,"[medals, moguls, sports, team, funding, olympi...","[medals, moguls, funding, aerial, sports, olym...",NaN,NaN
8,‘Put the blinders on’: how Jakara Anthony can ...,"[anthony, moguls, says, ski, win, field, seaso...","[anthony, says, moguls, field, makes, woman, s...",NaN,NaN
9,Ice in his veins: Australian skier Cooper Wood...,"[woods, said, just, cooper, lot, final, win, t...","[woods, cooper, said, just, lot, perisher, nsw...",NaN,NaN


In [16]:
# Compare Counts and TF/IDF

# Sample 5 random articles
samples = random.sample(range(0,len(terms_df)),5)

for sample in samples:
    doc = terms_df.iloc[sample]
    print(f"[{sample}] {doc['title']}")
    print("\t>> Counts:\t",doc['count'])
    print("\t>> TFIDF:\t",doc['tfidf'])
    print()

[12] Morning Mail: Taylor sets up Liberal spill, millionaires call for higher taxes, Australia’s skiing hope slips up [2026-02-11T19:40:12Z]
	>> Counts:	 ['morning', 'smith', 'shock', 'australians', 'today', 'able', 'sydney', 'set', 'action', 'week']
	>> TFIDF:	 ['morning', 'shock', 'smith', 'today', 'australians', 'set', 'cabinet', 'shadow', 'update', 'queensland']

[2] ‘Even more special’: Jakara Anthony dusts off Winter Olympics heartbreak for historic triumph [2026-02-15T01:32:16Z]
	>> Counts:	 ['moguls', 'anthony', 'dual', 'single', 'event', 'just', 'final', 'time', 'course', 'lot']
	>> TFIDF:	 ['moguls', 'anthony', 'dual', 'single', 'course', 'event', 'just', 'needed', 'special', 'help']

[10] Australia’s Jakara Anthony clinches first ever dual moguls Olympics gold [2026-02-14T11:40:32Z]
	>> Counts:	 ['anthony', 'second', 'moguls', 'james', 'halfpipe', 'training', 'event', 'won', 'peel', 'ok']
	>> TFIDF:	 ['anthony', 'ok', 'second', 'peel', 'james', 'moguls', 'singles', 'beating'

### Topic modelling with Latent Dirichlet Allocation (LDA)

[LDA](https://en.wikipedia.org/wiki/Latent_Dirichlet_allocation) is an algorithm for obtaining *topics* (a list of terms) from a document-term matrix. It is a generative probabilistic approach to *decomposition* of the document-term matrix into 2 factor matrices: document-topic and topic-term.

![img](https://editor.analyticsvidhya.com/uploads/26864dtm.JPG)

*Source: [Analytics Vidhya](https://www.analyticsvidhya.com/blog/2021/06/part-2-topic-modeling-and-latent-dirichlet-allocation-lda-using-gensim-and-sklearn/)*

The LDA model requires the number of topics to be set in advance. As it is a generative model, it also runs over a number of iterations. These values usually need to be experimented with to obtain quality topics.

In [17]:
# Set number of topics
num_topics = 15
# Set max number of iteractions
max_iterations = 20

# Create the model
lda_model = LatentDirichletAllocation(n_components=num_topics,max_iter=max_iterations,learning_method='online')

# Fit the model to the data, and use the model to transform the data (do the decomposition)
doc_topic_matrix = lda_model.fit_transform(count_dt_matrix)

# Obtain the topics
topic_term_matrix = lda_model.components_

#### View the topics

In [18]:
# Get the topics and their terms
lda_topic_dict = {}
for index, topic in enumerate(topic_term_matrix):
    zipped = zip(feature_names, topic)
    top_terms=dict(sorted(zipped, key = lambda t: t[1], reverse=True)[:10])
    #print(top_terms)
    top_terms_list= {key : round(top_terms[key], 4) for key in top_terms.keys()}
    lda_topic_dict[f"topic_{index}"] = top_terms_list

# Print the topics with their terms    
for k,v in lda_topic_dict.items():
    print(k)
    print(v)
    print()

topic_0
{'anthony': np.float64(0.1747), 'moguls': np.float64(0.1659), 'medals': np.float64(0.1522), 'rounds': np.float64(0.1492), 'support': np.float64(0.1491), 'sports': np.float64(0.1491), 'northern': np.float64(0.1478), 'judges': np.float64(0.1478), 'haul': np.float64(0.1476), 'yellow': np.float64(0.147)}

topic_1
{'just': np.float64(17.7741), 'time': np.float64(13.0911), 'skiing': np.float64(12.1719), 'indra': np.float64(12.1524), 'medals': np.float64(11.2192), 'team': np.float64(10.3238), 'brown': np.float64(9.3795), 'halfpipe': np.float64(9.3725), 'ski': np.float64(9.3684), 'moguls': np.float64(8.4763)}

topic_2
{'anthony': np.float64(0.2468), 'moguls': np.float64(0.2164), 'final': np.float64(0.185), 'event': np.float64(0.1745), 'single': np.float64(0.1744), 'big': np.float64(0.167), 'win': np.float64(0.1659), 'dual': np.float64(0.1637), 'team': np.float64(0.1636), 'just': np.float64(0.1623)}

topic_3
{'bradbury': np.float64(0.2197), 'event': np.float64(0.2127), 'high': np.float6

#### List of topics for each document

In [19]:
doc_topic_matrix

array([[2.64550266e-04, 9.96296294e-01, 2.64550267e-04, 2.64550267e-04,
        2.64550266e-04, 2.64550778e-04, 2.64550266e-04, 2.64550667e-04,
        2.64550266e-04, 2.64550504e-04, 2.64550584e-04, 2.64550266e-04,
        2.64551248e-04, 2.64550268e-04, 2.64550267e-04],
       [2.29095076e-04, 2.29095556e-04, 2.29095077e-04, 2.29095076e-04,
        2.29095076e-04, 2.29095548e-04, 2.29095076e-04, 2.29095607e-04,
        2.29095076e-04, 2.29095256e-04, 2.29095417e-04, 2.29095076e-04,
        9.96792667e-01, 2.29095076e-04, 2.29095076e-04],
       [2.76625175e-04, 2.76625810e-04, 2.76625176e-04, 2.76625175e-04,
        2.76625175e-04, 2.76625492e-04, 2.76625174e-04, 9.96127245e-01,
        2.76625174e-04, 2.76625433e-04, 2.76625703e-04, 2.76625174e-04,
        2.76625684e-04, 2.76625175e-04, 2.76625175e-04],
       [2.86123034e-04, 2.86123497e-04, 2.86123035e-04, 2.86123035e-04,
        2.86123035e-04, 2.86123297e-04, 2.86123034e-04, 2.86123398e-04,
        2.86123034e-04, 2.86123306e-0

#### Update the terms matrix

In [20]:
for idx,topic in enumerate(doc_topic_matrix):
    topic_num = topic.argmax()
    top_topic = lda_topic_dict[f"topic_{topic_num}"]
    terms_df.at[idx,'lda'] = list(top_topic.keys())

terms_df

,title,count,tfidf,lda,nmf
0,Winter Olympics 2026: what you need to know if...,"[2026, ice, new, athletes, aedt, day, ski, wat...","[ice, aedt, watch, 2026, athletes, table, moun...","[just, time, skiing, indra, medals, team, brow...",NaN
1,Australia’s upward trajectory slips off course...,"[team, anthony, final, competition, medals, tr...","[anthony, team, competition, air, snowboarder,...","[hamilton, ski, anthony, team, just, competiti...",NaN
2,‘Even more special’: Jakara Anthony dusts off ...,"[moguls, anthony, dual, single, event, just, f...","[moguls, anthony, dual, single, course, event,...","[anthony, moguls, second, dual, event, win, co...",NaN
3,A part-time job and DJ gigs helped Lara Hamilt...,"[hamilton, ski, skimo, just, running, says, sk...","[hamilton, skimo, running, ski, mountaineering...","[hamilton, ski, anthony, team, just, competiti...",NaN
4,Valentino Guseli’s unexpected big air dream en...,"[final, guseli, said, time, score, air, best, ...","[guseli, final, air, 50, landing, score, said,...","[final, guseli, said, time, big, air, best, sc...",NaN
5,From Bradbury to Bright: five of Australia’s b...,"[bradbury, event, high, bronze, skiing, way, s...","[bradbury, smith, high, bronze, city, event, s...","[woods, just, said, skiing, bradbury, lot, win...",NaN
6,Australia’s youngest Winter Olympian Indra Bro...,"[just, indra, brown, time, says, want, halfpip...","[indra, want, just, brown, says, ve, time, fam...","[just, time, skiing, indra, medals, team, brow...",NaN
7,How did Australia – better known for its beach...,"[medals, moguls, sports, team, funding, olympi...","[medals, moguls, funding, aerial, sports, olym...","[just, time, skiing, indra, medals, team, brow...",NaN
8,‘Put the blinders on’: how Jakara Anthony can ...,"[anthony, moguls, says, ski, win, field, seaso...","[anthony, says, moguls, field, makes, woman, s...","[anthony, moguls, second, dual, event, win, co...",NaN
9,Ice in his veins: Australian skier Cooper Wood...,"[woods, said, just, cooper, lot, final, win, t...","[woods, cooper, said, just, lot, perisher, nsw...","[woods, just, said, skiing, bradbury, lot, win...",NaN


#### Compare approaches

In [21]:
# Sample 5 random articles
samples = random.sample(range(0,len(terms_df)),5)

for sample in samples:
    doc = terms_df.iloc[sample]
    print(f"[{sample}] {doc['title']}")
    print("\t>> Counts:\t",doc['count'])
    print("\t>> TFIDF:\t",doc['tfidf'])
    print("\t>> LDA:\t\t",doc['lda'])
    print()

[12] Morning Mail: Taylor sets up Liberal spill, millionaires call for higher taxes, Australia’s skiing hope slips up [2026-02-11T19:40:12Z]
	>> Counts:	 ['morning', 'smith', 'shock', 'australians', 'today', 'able', 'sydney', 'set', 'action', 'week']
	>> TFIDF:	 ['morning', 'shock', 'smith', 'today', 'australians', 'set', 'cabinet', 'shadow', 'update', 'queensland']
	>> LDA:		 ['just', 'time', 'skiing', 'indra', 'medals', 'team', 'brown', 'halfpipe', 'ski', 'moguls']

[6] Australia’s youngest Winter Olympian Indra Brown: ‘I just love the feeling of flying’ | Martin Pegan [2026-02-01T14:00:34Z]
	>> Counts:	 ['just', 'indra', 'brown', 'time', 'says', 'want', 'halfpipe', 've', 'family', 'ski']
	>> TFIDF:	 ['indra', 'want', 'just', 'brown', 'says', 've', 'time', 'family', 'trick', 'landing']
	>> LDA:		 ['just', 'time', 'skiing', 'indra', 'medals', 'team', 'brown', 'halfpipe', 'ski', 'moguls']

[10] Australia’s Jakara Anthony clinches first ever dual moguls Olympics gold [2026-02-14T11:40:3

### Topic modelling with Non-negative Matrix Factorisation (NMF)


[NMF](https://en.wikipedia.org/wiki/Latent_Dirichlet_allocation) is a different algorithm for obtaining *topics* (a list of terms) from a document-term matrix. It also factorises the document-term matrix into 2 factor matrices: document-topic and topic-term.

In [22]:
# Set the number of topics
num_topics = 15

# Create the model
nmf_model = NMF(n_components=num_topics,init='random',beta_loss='frobenius')

# Fit the model to the data and use it to transform the data
doc_topic_nmf = nmf_model.fit_transform(tfidf_dt_matrix)

topic_term_nmf = nmf_model.components_

In [23]:
# Get the topics and their terms
nmf_topic_dict = {}
for index, topic in enumerate(topic_term_nmf):
    zipped = zip(feature_names, topic)
    top_terms=dict(sorted(zipped, key = lambda t: t[1], reverse=True)[:10])
    #print(top_terms)
    top_terms_list= {key : round(top_terms[key], 4) for key in top_terms.keys()}
    nmf_topic_dict[f"topic_{index}"] = top_terms_list

# Print the topics with their terms    
for k,v in nmf_topic_dict.items():
    print(k)
    print(v)
    print()

topic_0
{'moguls': np.float64(1.6981), 'anthony': np.float64(1.5484), 'dual': np.float64(0.9978), 'single': np.float64(0.9937), 'course': np.float64(0.61), 'event': np.float64(0.6096), 'just': np.float64(0.5211), 'help': np.float64(0.5001), 'needed': np.float64(0.5001), 'rounds': np.float64(0.5001)}

topic_1
{'anthony': np.float64(0.6397), 'ok': np.float64(0.4831), 'second': np.float64(0.472), 'peel': np.float64(0.4284), 'james': np.float64(0.3954), 'moguls': np.float64(0.3388), 'beating': np.float64(0.3221), 'ruptured': np.float64(0.3221), 'singles': np.float64(0.3221), 'likely': np.float64(0.3221)}

topic_2
{'hamilton': np.float64(1.4704), 'skimo': np.float64(0.9357), 'running': np.float64(0.8298), 'ski': np.float64(0.5626), 'mountaineering': np.float64(0.5347), 'says': np.float64(0.4924), 'just': np.float64(0.4923), 'results': np.float64(0.3556), 'skis': np.float64(0.3556), 'skiing': np.float64(0.3516)}

topic_3
{'indra': np.float64(2.0761), 'want': np.float64(1.5592), 'just': np.fl

#### Update the terms matrix

In [24]:
for idx,topic in enumerate(doc_topic_nmf):
    topic_num = topic.argmax()
    top_topic = nmf_topic_dict[f"topic_{topic_num}"]
    terms_df.at[idx,'nmf'] = list(top_topic.keys())

terms_df

,title,count,tfidf,lda,nmf
0,Winter Olympics 2026: what you need to know if...,"[2026, ice, new, athletes, aedt, day, ski, wat...","[ice, aedt, watch, 2026, athletes, table, moun...","[just, time, skiing, indra, medals, team, brow...","[ice, aedt, watch, 2026, athletes, mountaineer..."
1,Australia’s upward trajectory slips off course...,"[team, anthony, final, competition, medals, tr...","[anthony, team, competition, air, snowboarder,...","[hamilton, ski, anthony, team, just, competiti...","[anthony, competition, snowboarder, 000, air, ..."
2,‘Even more special’: Jakara Anthony dusts off ...,"[moguls, anthony, dual, single, event, just, f...","[moguls, anthony, dual, single, course, event,...","[anthony, moguls, second, dual, event, win, co...","[moguls, anthony, dual, single, course, event,..."
3,A part-time job and DJ gigs helped Lara Hamilt...,"[hamilton, ski, skimo, just, running, says, sk...","[hamilton, skimo, running, ski, mountaineering...","[hamilton, ski, anthony, team, just, competiti...","[hamilton, skimo, running, ski, mountaineering..."
4,Valentino Guseli’s unexpected big air dream en...,"[final, guseli, said, time, score, air, best, ...","[guseli, final, air, 50, landing, score, said,...","[final, guseli, said, time, big, air, best, sc...","[guseli, final, air, 50, landing, score, said,..."
5,From Bradbury to Bright: five of Australia’s b...,"[bradbury, event, high, bronze, skiing, way, s...","[bradbury, smith, high, bronze, city, event, s...","[woods, just, said, skiing, bradbury, lot, win...","[bradbury, smith, high, bronze, city, event, l..."
6,Australia’s youngest Winter Olympian Indra Bro...,"[just, indra, brown, time, says, want, halfpip...","[indra, want, just, brown, says, ve, time, fam...","[just, time, skiing, indra, medals, team, brow...","[indra, want, just, brown, says, ve, time, fam..."
7,How did Australia – better known for its beach...,"[medals, moguls, sports, team, funding, olympi...","[medals, moguls, funding, aerial, sports, olym...","[just, time, skiing, indra, medals, team, brow...","[medals, moguls, funding, aerial, sports, olym..."
8,‘Put the blinders on’: how Jakara Anthony can ...,"[anthony, moguls, says, ski, win, field, seaso...","[anthony, says, moguls, field, makes, woman, s...","[anthony, moguls, second, dual, event, win, co...","[says, field, season, woman, january, 20, wins..."
9,Ice in his veins: Australian skier Cooper Wood...,"[woods, said, just, cooper, lot, final, win, t...","[woods, cooper, said, just, lot, perisher, nsw...","[woods, just, said, skiing, bradbury, lot, win...","[woods, cooper, said, just, lot, man, nsw, per..."


### Compare approaches

In [25]:
# Sample 5 random articles
samples = random.sample(range(0,len(terms_df)),5)

for sample in samples:
    doc = terms_df.iloc[sample]
    print(f"[{sample}] {doc['title']}")
    print("\t>> Counts:\t",doc['count'])
    print("\t>> TFIDF:\t",doc['tfidf'])
    print("\t>> LDA:\t\t",doc['lda'])
    print("\t>> NMF:\t\t",doc['nmf'])
    print()

[0] Winter Olympics 2026: what you need to know if following from Australia [2026-02-02T14:00:10Z]
	>> Counts:	 ['2026', 'ice', 'new', 'athletes', 'aedt', 'day', 'ski', 'watch', 'skiing', 'time']
	>> TFIDF:	 ['ice', 'aedt', 'watch', '2026', 'athletes', 'table', 'mountaineering', 'hours', 'new', 'italy']
	>> LDA:		 ['just', 'time', 'skiing', 'indra', 'medals', 'team', 'brown', 'halfpipe', 'ski', 'moguls']
	>> NMF:		 ['ice', 'aedt', 'watch', '2026', 'athletes', 'mountaineering', 'table', 'hours', 'new', 'italy']

[6] Australia’s youngest Winter Olympian Indra Brown: ‘I just love the feeling of flying’ | Martin Pegan [2026-02-01T14:00:34Z]
	>> Counts:	 ['just', 'indra', 'brown', 'time', 'says', 'want', 'halfpipe', 've', 'family', 'ski']
	>> TFIDF:	 ['indra', 'want', 'just', 'brown', 'says', 've', 'time', 'family', 'trick', 'landing']
	>> LDA:		 ['just', 'time', 'skiing', 'indra', 'medals', 'team', 'brown', 'halfpipe', 'ski', 'moguls']
	>> NMF:		 ['indra', 'want', 'just', 'brown', 'says', 

### How should the 'best' approach be selected?

Keep in mind:

* Count and TF/IDF are deterministic, but LDA and NMF are stochastic
* Stochastic algorithms will always need some experimentation, and will change in effectiveness based on the kind of data that is being analysed
* As with all data analytics, we need to make decisions based on the Question and the context in which the question is asked. This means also considering the stakeholders
*

